In [3]:
from newsapi import NewsApiClient
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install setuptools
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews
from tqdm import tqdm
import requests
import json

import warnings
warnings.filterwarnings('ignore')


[==================================================] 100.0% 104.8/104.8MB downloaded


# Get S&P500 companies' tickers

In [4]:
response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
companies_info_df = pd.read_html(response.content)[0]
companies_info_df = companies_info_df.drop(columns=["No.", "Stock Price", "% Change", "Revenue"])
companies_info_df.to_excel("Data/SPY_companies_info.xlsx", index=False)

In [5]:
# tech stocks, pharma stocks, oil stocks, tobacco stocks and Market
tickers = companies_info_df["Symbol"].to_list()

# News Data Download
- newsapi has a limit of 2024-11-01 onwards
- pygoognews https://github.com/kotartemiy/pygooglenews
- Some other packages to try out: https://www.newscatcherapi.com/blog/python-web-scraping-libraries-to-mine-news-data

In [ ]:
# Looping over 500 tickers: this run will take a few hours
gn = GoogleNews(lang = 'en')

news_dfs = []
for ticker in tqdm(tickers):
    top = gn.search(ticker)
    entries = top["entries"]
    df_temp = clean_goog_news(entries)
    df_temp["date"] = df_temp["date"].apply(lambda d: pd.to_datetime(d, errors='coerce').date())
    df_temp = df_temp.dropna()
    news_dfs += [df_temp.copy()]


In [6]:
news_dfs[0]

,date,title,source
0,"Wed, 04 Dec 2024 10:40:07 GMT",AAPL Stock Hits New High Despite Sales Slowdow...,https://news.google.com/rss/articles/CBMilwJBV...
1,"Fri, 06 Dec 2024 18:04:28 GMT",Why Apple Inc. (AAPL) Is the Best Beginner Sto...,https://news.google.com/rss/articles/CBMieEFVX...
2,"Fri, 06 Dec 2024 19:37:00 GMT",Best Dow Jones Stocks To Watch In December 202...,https://news.google.com/rss/articles/CBMidEFVX...
3,"Fri, 06 Dec 2024 15:55:00 GMT",Why Apple Doesn't Need Nvidia (NASDAQ:AAPL) - ...,https://news.google.com/rss/articles/CBMieEFVX...
4,"Wed, 04 Dec 2024 06:38:01 GMT",Apple (AAPL) Faces $1B Class Action Lawsuit Ov...,https://news.google.com/rss/articles/CBMingFBV...
...,...,...,...
94,"Wed, 04 Dec 2024 08:17:12 GMT",Thomasville National Bank Increases Stock Hold...,https://news.google.com/rss/articles/CBMixwFBV...
95,"Wed, 04 Dec 2024 16:47:28 GMT",Apple (NASDAQ:AAPL) Shares Up 0.3% Following A...,https://news.google.com/rss/articles/CBMirgFBV...
96,"Wed, 04 Dec 2024 08:17:19 GMT","Granite Harbor Advisors Inc. Buys 3,697 Shares...",https://news.google.com/rss/articles/CBMivwFBV...
97,"Tue, 26 Nov 2024 08:00:00 GMT",Apple Inc. (AAPL): Developing Advanced LLM Sir...,https://news.google.com/rss/articles/CBMihwFBV...


## General News 
- Newscatcher for general news not targetted at specific stocks: https://github.com/kotartemiy/newscatcher


In [ ]:
topics = ['tech', 'news', 'business', 'science', 'finance', 'food', 'politics', 'economics', 'travel', 'entertainment', 'music', 'sport', 'world']
all_general_news = pd.DataFrame()

for t in tqdm(topics):
    news_temp = clean_newscatcher_news(t)
    all_general_news = pd.concat([all_general_news, news_temp])



  0%|          | 0/13 [00:00<?, ?it/s]

No. of URLs: 163
['facebook.com', 'twitter.com', 'theguardian.com', 'cnet.com', 'wired.com', 'digg.com', 'samsung.com', 'theverge.com', 'mashable.com', 'engadget.com', 'businesswire.com', 'lifehacker.com', 'utoronto.ca', 'eff.org', 'theregister.co.uk', 'pcworld.com', 'prweb.com', 'acm.org', 'techradar.com', 'ndtv.com', 'boingboing.net', 'popularmechanics.com', 'crunchbase.com', 'ycombinator.com', 'macrumors.com', 'cio.com', 'gigaom.com', 'ibtimes.co.uk', 'networkworld.com', 'extremetech.com', 'windows.com', 'dailydot.com', 'gsmarena.com', 'anandtech.com', 'slashgear.com', 'computerweekly.com', 'betanews.com', 'techdirt.com', 'houstonchronicle.com', 'techtimes.com', 'androidcentral.com', 'techspot.com', 'infoq.com', 'macmillan.com', 'windowscentral.com', 'neowin.net', 'phonearena.com', 'tmcnet.com', 'pocket-lint.com', 'cbinsights.com', 'theiet.org', 'thewirecutter.com', 'gizmodo.com.au', 'nerdist.com', 'itworld.com', 'internetsociety.org', 'themarysue.com', 'laptopmag.com', 'wccftech.co

100%|██████████| 13/13 [56:31<00:00, 260.92s/it]  


In [135]:
all_general_news["date"] = all_general_news["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
all_general_news = all_general_news.dropna()
all_general_news.sort_values(by='date', inplace=True)
all_general_news = all_general_news.reset_index(drop=True)

all_general_news

,date,title,source,topic
0,1970-01-01,DAFT.IE PROPERTY MAGAZINE,https://www.thejournal.ie/river/daft-ie-proper...,news
1,2008-08-01,The World's Oldest Jokes Revealed,https://www.sciencecodex.com/worlds-oldest-jok...,science
2,2008-10-18,Charlie Gasparino on Barack Obama,https://www.realclearmarkets.com/,finance
3,2008-10-18,Was Deregulation The Problem?,https://www.realclearmarkets.com/,finance
4,2008-10-18,Is It a Liquidity or Solvency Crisis?,https://www.realclearmarkets.com/,finance
...,...,...,...,...
28930,2024-12-09,VARIOUS - 2024: Past/Present Compilation (Past...,https://www.juno.co.uk/products/2024-past-pres...,music
28931,2024-12-09,NITECHORD - Lume (Past Inside The Present US) ...,https://www.juno.co.uk/products/nitechord-lume...,music
28932,2024-12-09,"Where Tyranny Begins: The Justice Department, ...",https://www.newamerica.org/future-security/eve...,news
28933,2024-12-10,Could US tariffs ramp-up deflationary forces i...,https://www.investing.com/news/economy/could-u...,finance


In [ ]:
with pd.ExcelWriter('Data/news_data.xlsx') as writer:
    for i, news_df in tqdm(enumerate(news_dfs)):
        news_df.to_excel(writer, sheet_name=tickers[i], index=False)
    all_general_news.to_excel(writer, sheet_name="General", index=False)
# Close the ExcelWriter object
writer.save()

# CapIQ Data
Data is acquired from SMU Library Website https://researchguides.smu.edu.sg/az.php?a=c

In [11]:
news = pd.ExcelFile('./Data/news_data.xlsx')
news_dict = {
    sheet_name: news.parse(sheet_name) for sheet_name in news.sheet_names if sheet_name != 'General'
}

all_general_news = pd.read_excel('./Data/news_data.xlsx', sheet_name="General")


In [14]:
capiq_df = pd.read_excel("Data/capiq_news_data.xlsx")
capiq_df = capiq_df.rename(columns = {"Key Developments By Date": "date",
                            "Key Development Headline": "title",
                            "Key Development Sources": "source",
                            "Primary Industry": "topic"
                            })

capiq_df =  capiq_df.drop(columns=["Key Developments by Type", "Key Development Situation"])

In [4]:
capiq_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448274 entries, 0 to 448273
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   date                  448274 non-null  datetime64[ns]
 1   Company Name(s)       448274 non-null  object        
 2   title                 448274 non-null  object        
 3   source                448274 non-null  object        
 4   Business Description  448274 non-null  object        
 5   topic                 448274 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 20.5+ MB


In [15]:
# Get ticker
capiq_df["Ticker"] = capiq_df["Company Name(s)"].str.replace(r'[^(]*\(|\)[^)]*', '')
capiq_df["Ticker"] = capiq_df["Ticker"].str.split(':').str[-1]
capiq_df["Ticker"] = capiq_df["Ticker"].str.replace(r'[^a-zA-Z]+', '')

capiq_df

,date,Company Name(s),title,source,Business Description,topic,Ticker
0,2014-01-01,"Motorola Solutions, Inc. (NYSE:MSI)",Motorola Solutions to Provide IDF's Battlefiel...,Other,"Motorola Solutions, Inc. provides public safet...",Communications Equipment,MSI
1,2014-01-01,TE Connectivity plc (NYSE:TEL),TE Connectivity Brings Advanced Mobile Service...,Business Wire,"TE Connectivity plc, together with its subsidi...",Electronic Manufacturing Services,TEL
2,2014-01-01,"Leidos Holdings, Inc. (NYSE:LDOS)",Leidos Holdings Receives Follow-On Contract fr...,Datamonitor NewsWire,"Leidos Holdings, Inc., together with its subsi...",Research and Consulting Services,LDOS
3,2014-01-01,MarketAxess Holdings Inc. (NasdaqGS:MKTX),MarketAxess Holdings Inc.'s Equity Buyback ann...,Capital IQ Buybacks Database,"MarketAxess Holdings Inc., together with its s...",Financial Exchanges and Data,MKTX
4,2014-01-01,Amgen Inc. (NasdaqGS:AMGN); UCB SA (ENXTBR:UCB),Amgen and UCB Announces Results from Phase 2 T...,PR Newswire,Amgen Inc. (NasdaqGS:AMGN)Amgen Inc. discovers...,Amgen Inc. (NasdaqGS:AMGN) (Biotechnology); UC...,UCB
...,...,...,...,...,...,...,...
448269,2024-12-04,CSX Corporation (NasdaqGS:CSX),CSX Corporation Presents at UBS Global Industr...,PR Newswire; Business Wire; GlobeNewswire; Com...,"CSX Corporation, together with its subsidiarie...",Rail Transportation,CSX
448270,2024-12-04,UnitedHealth Group Incorporated (NYSE:UNH),UnitedHealth Group Incorporated - Analyst/Inve...,Business Wire,UnitedHealth Group Incorporated operates as a ...,Managed Health Care,UNH
448271,2024-12-04,LyondellBasell Industries N.V. (NYSE:LYB),LyondellBasell Industries N.V. Presents at Gol...,PR Newswire; Business Wire; GlobeNewswire; Com...,LyondellBasell Industries N.V. operates as a c...,Commodity Chemicals,LYB
448272,2024-12-04,Freeport-McMoRan Inc. (NYSE:FCX),Freeport-McMoRan Inc. Presents at Mines and Mo...,Company Website,Freeport-McMoRan Inc. engages in the mining of...,Copper,FCX


## Merging CapIQ, PyGoogleNews and NewsCatcher

- CapIQ and PyGoogleNews will be concatenated based on the 'ticker' column
- CapIQ and NewsCatcher will be concatenated based on the 'topic' column 
    * NewsCatcher has less and defined topics for each news. Hence, 'glove-twitter-25' is used to evaluate which topic for each stock in CapIQ has the closest similarity to the topic in NewsCatcher
    * Note that 'glove-twitter-25' required both texts to have the same number of words to evaluate its similarity, hence 7 random non-NA rows of the first word of the topic for each stock in CapIQ will be evaluated against each topic found in NewsCatcher

In [16]:
capiq_df_tickers = capiq_df["Ticker"].unique().tolist()
found_tickers = 0
for t in tqdm(capiq_df_tickers):
    news_temp = capiq_df[capiq_df["Ticker"] == t].copy()
    if t in news_dict:
        curr_news_df = news_dict[t].copy()
        match_topic_newscatcher = get_topic(news_temp["Business Description"].dropna()) # already sorted by ticker, just get the first non-NA topic
        curr_newscatcher = all_general_news[all_general_news["topic"].str.contains(match_topic_newscatcher)].copy()
        news_dict[t] = pd.concat([curr_news_df, news_temp.drop(columns=["Ticker", "Company Name(s)", "Business Description"]), curr_newscatcher])
        news_dict[t]["date"] = news_dict[t]["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
        news_dict[t] = news_dict[t].dropna()
        news_dict[t].sort_values(by='date', inplace=True)
        news_dict[t] = news_dict[t].reset_index(drop=True)
        found_tickers += 1
    else:
        continue
print(f"Found tickers: {found_tickers}")

100%|██████████| 3917/3917 [16:44<00:00,  3.90it/s] 

Found tickers: 496


In [17]:
news_dict_temp = dict(news_dict)
for t, news in news_dict_temp.items():
    if t not in capiq_df_tickers:
        del news_dict[t]

In [18]:
# separate into 3 files because github only allows files <100,000mb to be pushed

tickers = list(news_dict.keys())
dfs = list(news_dict.values())

with pd.ExcelWriter(f'Data/news_data_final1.xlsx') as writer:
    for i in tqdm(range(165)):
        dfs[i] = dfs[i].drop_duplicates()
        dfs[i].to_excel(writer, sheet_name=tickers[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final2.xlsx') as writer:
    for i in tqdm(range(165, 330)):
        dfs[i] = dfs[i].drop_duplicates()
        dfs[i].to_excel(writer, sheet_name=tickers[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final3.xlsx') as writer:
    for i in tqdm(range(330,len(dfs))):
        dfs[i] = dfs[i].drop_duplicates()
        dfs[i].to_excel(writer, sheet_name=tickers[i], index=False)

100%|██████████| 166/166 [01:29<00:00,  1.85it/s]


# Train Test Split

In [24]:
news_dict_train = {}
news_dict_test = {}

for ticker, news_df in news_dict.items():
    news_df["date"] = pd.to_datetime(news_df["date"])
    news_dict_train[ticker] = news_df[news_df["date"].dt.year < 2024].copy()
    news_dict_test[ticker] = news_df[news_df["date"].dt.year >= 2024].copy()

In [ ]:
tickers_train = list(news_dict_train.keys())
dfs_train = list(news_dict_train.values())

tickers_test = list(news_dict_test.keys())
dfs_test = list(news_dict_test.values())

with pd.ExcelWriter(f'Data/news_data_final_rain1.xlsx') as writer:
    for i in tqdm(range(248)):
        dfs_train[i] = dfs_train[i].drop_duplicates()
        dfs_train[i].to_excel(writer, sheet_name=tickers_train[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final_train2.xlsx') as writer:
    for i in tqdm(range(248,len(tickers_train))):
        dfs_train[i] = dfs_train[i].drop_duplicates()
        dfs_train[i].to_excel(writer, sheet_name=tickers_train[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final_test1.xlsx') as writer:
    for i in tqdm(range(300)):
        dfs_test[i] = dfs_test[i].drop_duplicates()
        dfs_test[i].to_excel(writer, sheet_name=tickers_test[i], index=False)


with pd.ExcelWriter(f'Data/news_data_final_test2.xlsx') as writer:
    for i in tqdm(range(300,len(tickers_test))):
        dfs_test[i] = dfs_test[i].drop_duplicates()
        dfs_test[i].to_excel(writer, sheet_name=tickers_test[i], index=False)


100%|██████████| 196/196 [01:51<00:00,  1.76it/s]


# Removing Similar News


In [26]:
for ticker, df in tqdm(news_dict_train.items()):
    news_dict_train[ticker] = remove_similar_news(news_dict_train[ticker], "title", threshold=0.85)
    news_dict_train[ticker] = news_dict_train[ticker].reset_index(drop=True)
    

In [31]:
for ticker, df in tqdm(news_dict_test.items()):
    all_dates = news_dict_test[ticker]['date'].unique()
    test_news_filtered_df = pd.DataFrame()
    for d in all_dates:
        test_temp_df = news_dict_test[ticker][news_dict_test[ticker]['date']==d].copy()
        test_temp_df = remove_similar_news(test_temp_df, "title", threshold=0.85)
        test_news_filtered_df = pd.concat([test_news_filtered_df, test_temp_df])
    test_news_filtered_df = test_news_filtered_df.reset_index(drop=True)
    news_dict_test[ticker] = test_news_filtered_df.copy()

100%|██████████| 496/496 [16:48<00:00,  2.03s/it]


In [ ]:
tickers_train = list(news_dict_train.keys())
dfs_train = list(news_dict_train.values())

tickers_test = list(news_dict_test.keys())
dfs_test = list(news_dict_test.values())

with pd.ExcelWriter(f'Data/news_data_final_filtered_train1.xlsx') as writer:
    for i in tqdm(range(248)):
        dfs_train[i] = dfs_train[i].drop_duplicates()
        dfs_train[i].to_excel(writer, sheet_name=tickers_train[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final_filtered_train2.xlsx') as writer:
    for i in tqdm(range(248,len(tickers_train))):
        dfs_train[i] = dfs_train[i].drop_duplicates()
        dfs_train[i].to_excel(writer, sheet_name=tickers_train[i], index=False)

with pd.ExcelWriter(f'Data/news_data_final_filtered_test1.xlsx') as writer:
    for i in tqdm(range(300)):
        dfs_test[i] = dfs_test[i].drop_duplicates()
        dfs_test[i].to_excel(writer, sheet_name=tickers_test[i], index=False)


with pd.ExcelWriter(f'Data/news_data_final_filtered_stest2.xlsx') as writer:
    for i in tqdm(range(300,len(tickers_test))):
        dfs_test[i] = dfs_test[i].drop_duplicates()
        dfs_test[i].to_excel(writer, sheet_name=tickers_test[i], index=False)

100%|██████████| 196/196 [01:54<00:00,  1.71it/s]


# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [6]:
stocks = yf.download(tickers, threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-01)')


In [7]:
stocks

Ticker            SYF                                                         \
Price            Open       High        Low      Close  Adj Close     Volume   
Date                                                                           
2014-01-02        NaN        NaN        NaN        NaN        NaN        NaN   
2014-01-03        NaN        NaN        NaN        NaN        NaN        NaN   
2014-01-06        NaN        NaN        NaN        NaN        NaN        NaN   
2014-01-07        NaN        NaN        NaN        NaN        NaN        NaN   
2014-01-08        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2024-11-22  65.040001  66.250000  65.040001  65.989998  65.989998  3511500.0   
2024-11-25  66.480003  67.589996  66.279999  67.040001  67.040001  5898200.0   
2024-11-26  66.389999  67.309998  66.084999  67.260002  67.260002  2924100.0   
2024-11-27  67.419998  67.639999  66.459999  67.220001  67.220001  2692100.0   
2024-11-29  68.040001  68.690002  67.419998  67.519997  67.519997  1578300.0   

Ticker             ELV                                      ...         BG  \
Price             Open        High         Low       Close  ...        Low   
Date                                                        ...              
2014-01-02   92.000000   92.010002   90.620003   90.629997  ...  81.559998   
2014-01-03   90.459999   91.059998   90.309998   90.629997  ...  80.949997   
2014-01-06   91.019997   91.050003   89.250000   89.279999  ...  80.940002   
2014-01-07   89.669998   91.489998   89.419998   91.080002  ...  81.150002   
2014-01-08   90.839996   92.220001   90.669998   92.220001  ...  81.110001   
...                ...         ...         ...         ...  ...        ...   
2024-11-22  403.149994  406.730011  400.339996  402.549988  ...  86.900002   
2024-11-25  404.470001  409.769989  403.519989  407.600006  ...  87.669998   
2024-11-26  408.760010  408.760010  398.019989  401.959991  ...  88.029999   
2024-11-27  401.470001  407.730011  401.459991  402.750000  ...  88.559998   
2024-11-29  401.010010  409.750000  400.440002  406.959991  ...  88.500000   

Ticker                                            HON                          \
Price           Close  Adj Close   Volume        Open        High         Low   
Date                                                                            
2014-01-02  81.919998  60.954258   369100   86.718491   86.890099   85.850929   
2014-01-03  81.250000  60.455708   624700   86.270409   86.718491   86.117867   
2014-01-06  81.089996  60.336670  1249500   86.708961   87.004501   86.184608   
2014-01-07  81.650002  60.753349   968200   86.346680   86.890099   86.251343   
2014-01-08  81.330002  60.515274   695100   85.869995   86.565956   85.726990   
...               ...        ...      ...         ...         ...         ...   
2024-11-22  87.650002  87.650002  1611400  227.750000  230.119995  227.119995   
2024-11-25  88.440002  88.440002  1980800  231.119995  231.990005  229.699997   
2024-11-26  89.660004  89.660004  1518300  231.800003  232.500000  229.850006   
2024-11-27  88.910004  88.910004   851400  232.179993  232.960007  229.470001   
2024-11-29  89.739998  89.739998   801200  229.410004  233.270004  229.410004   

Ticker                                       
Price            Close   Adj Close   Volume  
Date                                         
2014-01-02   86.108337   68.652214  1829511  
2014-01-03   86.299011   68.804214  1544522  
2014-01-06   86.213203   68.735817  2051461  
2014-01-07   86.565956   69.017044  1650672  
2014-01-08   86.222740   68.743416  2774685  
...                ...         ...      ...  
2024-11-22  229.110001  229.110001  3834300  
2024-11-25  230.600006  230.600006  3837700  
2024-11-26  230.399994  230.399994  4203100  
2024-11-27  229.639999  229.639999  2870000  
2024-11-29  232.929993  232.929993  1917500  

[2747 

In [ ]:
# ticker_tz_df = pd.DataFrame()
# for t in tickers:
#     try:
#         ticker_info = yf.Ticker(t)
#         ticker_tz = ticker_info.info['timeZoneShortName']
#         ticker_tz_temp_df = pd.DataFrame({'Ticker':[t], 'tz':[ticker_tz]})
#         ticker_tz_df = pd.concat([ticker_tz_df, ticker_tz_temp_df])
#     except:
#         print(f'No timezone info for {t}')

No timezone info for BRK.B


In [ ]:
# print(f'Timezones of SPY stocks include: {ticker_tz_df.tz.unique()}')

Timezones of SPY stocks include: ['EST']


In [12]:
# Convert Stocks timezone to UTC 0
stocks.index = (
    stocks.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [13]:
market = yf.download(["SPY"], threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [14]:
market

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-01-02,183.979996,184.070007,182.479996,182.919998,151.242950,119636900
2014-01-03,183.229996,183.600006,182.630005,182.889999,151.218124,81390600
2014-01-06,183.490005,183.559998,182.080002,182.360001,150.779861,108028200
2014-01-07,183.089996,183.789993,182.949997,183.479996,151.705963,86144200
2014-01-08,183.449997,183.830002,182.889999,183.520004,151.739090,96582300
...,...,...,...,...,...,...
2024-11-22,593.659973,596.150024,593.150024,595.510010,595.510010,38226400
2024-11-25,599.520020,600.859985,595.200012,597.530029,597.530029,42441400
2024-11-26,598.799988,601.330017,598.070007,600.650024,600.650024,45621300


In [15]:
market.index = (
    market.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [16]:
stocks.to_excel("Data/stocks_data.xlsx")

In [17]:
market.to_excel("Data/market_data.xlsx")